In [3]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score
import pandas as pd
import numpy as np

In [4]:
# Generate some sample data
X = np.random.rand(100, 5)
y = 2 * X[:, 0] + 3 * X[:, 1] - 1.5 * X[:, 2] + np.random.randn(100) * 0.1

In [5]:
# Convert to pandas DataFrame for easier handling
X = pd.DataFrame(X)
y = pd.Series(y)

In [6]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

XGBoost provides many hyperparameters but we will only consider a few of them (see the [XGBoost documentation](https://xgboost.readthedocs.io/en/latest/parameter.html#general-parameters) for an complete overview). 

Note that we will use the [scikit-learn wrapper interface](https://xgboost.readthedocs.io/en/latest/python/python_api.html#module-xgboost.sklearn):


- `objective`: determines the loss function to be used like `reg:squarederror` for regression problems, `binary:logistic` for logistic regression for binary classification or `multi:softmax`  to do multiclass classification using the softmax objectiv (note that there are more options).


- `n_estimators`: Number of gradient boosted trees. Equivalent to number of boosting rounds.


- `max_depth`: Maximum tree depth for base learners.


- `learning_rate`: Boosting learning rate (xgb’s “eta”). Range is [0,1].


- `subsample`: Subsample ratio of the training instance. Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees. 


- `colsample_bytree`: Subsample ratio of columns when constructing each tree. Subsampling occurs once for every tree constructed.


- `colsample_bylevel`: Subsample ratio of columns for each level. Subsampling occurs once for every new depth level reached in a tree. Columns are subsampled from the set of columns chosen for the current tree.


- `gamma`: Minimum loss reduction required to make a further partition on a leaf node of the tree. A higher value leads to fewer splits. 


- `reg_alpha`: L1 regularization term on weights (xgb’s alpha). A large value leads to more regularization.


- `reg_lambda`: L2 regularization term on weights (xgb’s lambda).

- `eval_metric` (default is according to objective). Evaluation metrics for validation data, a default metric will be assigned according to objective (`rmse` for regression, and `logloss` for classificationestimator
).


- `random_state`: Random number seed.


- `early_stopping_rounds`: If you have a validation set, you can use early stopping to find the optimal number of boosting rounds. Early stopping requires at least one set in evals. If there’s more than one, it will use the last. Note that we include this parameter in our fit function.


In [7]:
# Initialize XGBoost regressor
xgbr = xgb.XGBRegressor(objective='reg:squarederror',
                            n_estimators=100,
                            learning_rate=0.1,
                            max_depth=5,
                            random_state=42)

In [8]:
# Train the model
xgbr.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [9]:
# Make predictions
y_pred = xgbr.predict(X_test)

In [10]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"r2score: {r2}")

Mean Squared Error: 0.14859754433060532
r2score: 0.8590685329850579


## Feature Importance

In [11]:
# make a pandas data series 
pd.Series(xgbr.feature_importances_, index=X.columns)

0    0.107724
1    0.621959
2    0.252169
3    0.005558
4    0.012590
dtype: float32

In [12]:
from sklearn.inspection import permutation_importance
# using permutation
result = permutation_importance(
    xgbr, X_test, y_test, n_repeats=10, random_state=42
)

In [13]:
pd.Series(result.importances_mean, index=X.columns)

0    0.414640
1    1.204929
2    0.128742
3    0.004168
4   -0.003144
dtype: float64

## When to Use Them

XGBoost is a great choice when:

- You have a structured dataset with labeled data.
- You need high accuracy and performance.
- You want to handle missing values automatically.
- You need to prevent overfitting.
- You have computational resources to train the model.

## Alternatives

Alternatives to XGBoost include:

- LightGBM: Another gradient boosting framework known for its speed and efficiency.
- CatBoost: Gradient boosting algorithm that handles categorical features well.
- Random Forest: Ensemble learning method based on decision trees.
- Gradient Boosting Machines (GBM): The general class of algorithms that XGBoost builds upon.

## Pros

Advantages of XGBoost:

- High Performance: Often achieves state-of-the-art results.
- Regularization: Helps prevent overfitting.
- Parallel Processing: Speeds up training.
- Handling Missing Values: Can automatically handle missing data.
- Feature Importance: Provides insights into feature importance.
- Flexibility: Can be used for both regression and classification tasks.

## Cons

Disadvantages of XGBoost:

- Complexity: Can be more complex to configure and tune compared to simpler algorithms.
- Overfitting: Can still overfit if not properly regularized.
- Computationally Intensive: Training can be computationally expensive, especially for large datasets.
- Black Box: Can be difficult to interpret the model's decision-making process.